# 🛒 Retail Fraud Detection with Machine Learning

---

## 📌 What is this notebook about?

Every day, thousands of **fraudulent transactions** happen in retail. Detecting fraud quickly saves businesses and customers from huge losses.

In this notebook, we will:
1. **Explore** the dataset (understand what's in it)
2. **Clean & prepare** the data for machine learning
3. **Train** a machine learning model to detect fraud
4. **Evaluate** how good our model is

> 💡 **Beginner tip:** Think of this as teaching a computer to spot suspicious transactions, just like a bank fraud team would!

---

## 📦 Dataset Overview

| Feature | Description |
|---|---|
| `transaction_amount` | How much money was spent |
| `payment_method` | Card, UPI, cash, etc. |
| `device_type` | Mobile, desktop, etc. |
| `is_international` | Was it a foreign transaction? |
| `previous_fraud_flag` | Has this customer committed fraud before? |
| `fraud_flag` | **Target** — 1 = Fraud, 0 = Legit |

## Step 1 — Import Libraries

These are the tools we need. Think of them as apps we install before starting work.

In [ ]:
# Data handling
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score
)

import warnings
warnings.filterwarnings('ignore')

# Make plots look nice
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 5)

print('✅ All libraries imported successfully!')

## Step 2 — Load the Dataset

We load the CSV file and take our **first look** at the data.

In [ ]:
df = pd.read_csv('/kaggle/input/retail-fraud-detection/retail_fraud_detection_100k.csv')

print(f'📊 Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# Check column data types
print('🔍 Column Info:')
df.info()

In [ ]:
# Check for missing values — important!
missing = df.isnull().sum()
print('❓ Missing values per column:')
print(missing[missing > 0] if missing.any() else '✅ No missing values found!')

In [ ]:
# Basic statistics about numerical columns
df.describe().round(2)

## Step 3 — Exploratory Data Analysis (EDA)

EDA means **visually exploring** our data to understand patterns before modelling.

In [ ]:
# ---- 3.1 Target Distribution ----
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
counts = df['fraud_flag'].value_counts()
axes[0].bar(['Legit (0)', 'Fraud (1)'], counts.values,
            color=['#2ecc71', '#e74c3c'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Fraud vs Legit Transactions', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 300, f'{v:,}\n({v/len(df)*100:.1f}%)',
                 ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=['Legit', 'Fraud'],
            colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Distribution', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()
print('💡 The dataset is fairly balanced — good for training!')

In [ ]:
# ---- 3.2 Transaction Amount: Fraud vs Legit ----
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Box plot
df.boxplot(column='transaction_amount', by='fraud_flag', ax=axes[0],
           patch_artist=True,
           boxprops=dict(facecolor='#3498db', color='navy'),
           medianprops=dict(color='red', linewidth=2))
axes[0].set_title('Transaction Amount by Fraud Flag')
axes[0].set_xlabel('Fraud Flag (0=Legit, 1=Fraud)')
plt.sca(axes[0]); plt.title('Transaction Amount Distribution')

# Histogram overlay
df[df['fraud_flag'] == 0]['transaction_amount'].hist(
    bins=50, ax=axes[1], alpha=0.6, color='#2ecc71', label='Legit')
df[df['fraud_flag'] == 1]['transaction_amount'].hist(
    bins=50, ax=axes[1], alpha=0.6, color='#e74c3c', label='Fraud')
axes[1].set_title('Transaction Amount Histogram')
axes[1].set_xlabel('Amount')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ---- 3.3 Categorical Features vs Fraud ----
cat_cols = ['payment_method', 'device_type', 'merchant_category']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col in zip(axes, cat_cols):
    fraud_rate = df.groupby(col)['fraud_flag'].mean().sort_values(ascending=False)
    bars = ax.barh(fraud_rate.index, fraud_rate.values * 100,
                   color=sns.color_palette('Set2', len(fraud_rate)))
    ax.set_title(f'Fraud Rate by {col.replace("_", " ").title()}',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Fraud Rate (%)')
    for bar, val in zip(bars, fraud_rate.values):
        ax.text(val * 100 + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val*100:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ---- 3.4 Correlation Heatmap (numerical features only) ----
num_df = df.select_dtypes(include='number')

plt.figure(figsize=(12, 7))
corr = num_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # hide upper triangle
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, vmin=-1, vmax=1,
            annot_kws={'size': 8})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('💡 Features closer to 1.0 or -1.0 with fraud_flag are most useful for prediction!')

In [ ]:
# ---- 3.5 Binary flag analysis ----
binary_flags = [
    'is_international', 'previous_fraud_flag', 'unusual_amount_flag',
    'unusual_location_flag', 'multiple_transactions_short_time',
    'high_risk_device_flag', 'velocity_flag'
]

fraud_rates = {col: df[df[col] == 1]['fraud_flag'].mean() * 100
               for col in binary_flags}

plt.figure(figsize=(10, 4))
colors = ['#e74c3c' if v > 60 else '#f39c12' if v > 40 else '#2ecc71'
          for v in fraud_rates.values()]
bars = plt.barh(list(fraud_rates.keys()),
                list(fraud_rates.values()), color=colors)
plt.xlabel('Fraud Rate when Flag = 1 (%)')
plt.title('How Much Each Risk Flag Predicts Fraud', fontsize=12, fontweight='bold')
for bar, val in zip(bars, fraud_rates.values()):
    plt.text(val + 0.5, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}%', va='center')
plt.tight_layout()
plt.show()

## Step 4 — Feature Engineering

We **create new features** and **convert text columns** to numbers that the model can understand.

In [ ]:
df_model = df.copy()

# 4.1 Extract time features from timestamp
df_model['transaction_timestamp'] = pd.to_datetime(df_model['transaction_timestamp'])
df_model['hour']        = df_model['transaction_timestamp'].dt.hour
df_model['day_of_week'] = df_model['transaction_timestamp'].dt.dayofweek   # 0=Mon, 6=Sun
df_model['is_weekend']  = (df_model['day_of_week'] >= 5).astype(int)
df_model['is_night']    = ((df_model['hour'] >= 22) | (df_model['hour'] <= 5)).astype(int)

# 4.2 Amount-based features
# How far is this transaction from the customer's usual spending?
df_model['amount_ratio'] = (
    df_model['transaction_amount'] / (df_model['avg_transaction_amount_7d'] + 1)
)

# 4.3 Aggregate risk score (sum of all binary risk flags)
flag_cols = [
    'is_international', 'previous_fraud_flag', 'unusual_amount_flag',
    'unusual_location_flag', 'multiple_transactions_short_time',
    'high_risk_device_flag', 'velocity_flag'
]
df_model['total_risk_score'] = df_model[flag_cols].sum(axis=1)

# 4.4 Encode categorical columns using Label Encoding
# Label Encoding converts text to numbers: e.g. ['cash','card'] -> [0, 1]
le = LabelEncoder()
for col in ['payment_method', 'device_type', 'location', 'merchant_category']:
    df_model[col] = le.fit_transform(df_model[col])

print('✅ Feature engineering done!')
print(f'New features added: hour, day_of_week, is_weekend, is_night, amount_ratio, total_risk_score')

## Step 5 — Prepare Data for Model Training

In [ ]:
# Drop columns that are NOT useful for prediction
drop_cols = ['transaction_id', 'customer_id', 'transaction_timestamp', 'fraud_risk']
df_model.drop(columns=drop_cols, inplace=True)

# Separate features (X) and target (y)
# X = everything the model sees; y = what it must predict
X = df_model.drop(columns=['fraud_flag'])
y = df_model['fraud_flag']

print(f'Features (X) shape : {X.shape}')
print(f'Target  (y) shape  : {y.shape}')
print(f'\nFeature list ({len(X.columns)}):')
print(X.columns.tolist())

In [ ]:
# Train-Test Split: 80% for training, 20% for testing
# random_state=42 makes results reproducible (same split every run)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training set   : {X_train.shape[0]:,} rows')
print(f'Testing  set   : {X_test.shape[0]:,} rows')

# Scale numerical features so they're on the same range
# (important for Logistic Regression; doesn't hurt tree-based models)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)   # only transform, never fit on test!

print('\n✅ Train-test split and scaling done!')

## Step 6 — Train Machine Learning Models

We try **3 models** and compare them. The best one wins! 🏆

In [ ]:
# Define our three models
models = {
    'Logistic Regression':     LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':           RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting':       GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = {}  # store results here

for name, model in models.items():
    print(f'\n⏳ Training {name}...')

    # Use scaled data for Logistic Regression, raw for tree models
    X_tr = X_train_scaled if name == 'Logistic Regression' else X_train
    X_te = X_test_scaled  if name == 'Logistic Regression' else X_test

    model.fit(X_tr, y_train)         # TRAIN
    y_pred  = model.predict(X_te)    # PREDICT (0 or 1)
    y_proba = model.predict_proba(X_te)[:, 1]  # probability of fraud

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    results[name] = {'model': model, 'y_pred': y_pred, 'y_proba': y_proba,
                     'accuracy': acc, 'roc_auc': auc}

    print(f'   Accuracy : {acc:.4f} ({acc*100:.2f}%)')
    print(f'   ROC-AUC  : {auc:.4f}')

print('\n✅ All models trained!')

## Step 7 — Model Comparison

In [ ]:
# Compare models side by side
comparison_df = pd.DataFrame({
    'Model'    : list(results.keys()),
    'Accuracy' : [v['accuracy'] for v in results.values()],
    'ROC-AUC'  : [v['roc_auc']  for v in results.values()]
}).sort_values('ROC-AUC', ascending=False)

print('📊 Model Comparison:')
print(comparison_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, metric in zip(axes, ['Accuracy', 'ROC-AUC']):
    colors = ['#2ecc71' if v == comparison_df[metric].max() else '#3498db'
              for v in comparison_df[metric]]
    bars = ax.bar(comparison_df['Model'], comparison_df[metric],
                  color=colors, edgecolor='white', linewidth=1.5)
    ax.set_ylim(comparison_df[metric].min() - 0.05, 1.02)
    ax.set_title(f'{metric} Comparison', fontsize=12, fontweight='bold')
    ax.set_ylabel(metric)
    for bar, val in zip(bars, comparison_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{val:.4f}', ha='center', fontweight='bold')
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

## Step 8 — Deep Dive: Best Model

We pick the **best performing model** and analyse it in depth.

In [ ]:
# Pick the best model by ROC-AUC
best_name = comparison_df.iloc[0]['Model']
best      = results[best_name]
print(f'🏆 Best Model: {best_name}')
print(f'   Accuracy : {best["accuracy"]*100:.2f}%')
print(f'   ROC-AUC  : {best["roc_auc"]:.4f}')

In [ ]:
# ---- 8.1 Classification Report ----
print('📋 Detailed Classification Report:')
print(classification_report(y_test, best['y_pred'],
                             target_names=['Legit (0)', 'Fraud (1)']))
# Precision = of all flagged fraud, how many were really fraud?
# Recall    = of all real fraud, how many did we catch?
# F1-score  = balance between precision and recall

In [ ]:
# ---- 8.2 Confusion Matrix ----
cm = confusion_matrix(y_test, best['y_pred'])
labels = ['True Legit\n(TN)', 'False Fraud\n(FP)', 'Missed Fraud\n(FN)', 'Caught Fraud\n(TP)']

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Legit', 'Predicted Fraud'],
            yticklabels=['Actual Legit', 'Actual Fraud'],
            linewidths=2, linecolor='white',
            annot_kws={'size': 14, 'weight': 'bold'})
plt.title(f'Confusion Matrix — {best_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (correctly said Legit)  : {tn:,}')
print(f'False Positives (wrongly flagged Fraud)  : {fp:,}')
print(f'False Negatives (missed real Fraud)      : {fn:,}')
print(f'True Positives  (correctly caught Fraud) : {tp:,}')

In [ ]:
# ---- 8.3 ROC Curves for ALL models ----
plt.figure(figsize=(8, 6))
colors_roc = ['#e74c3c', '#3498db', '#2ecc71']

for (name, res), color in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    plt.plot(fpr, tpr, color=color, lw=2,
             label=f"{name} (AUC = {res['roc_auc']:.4f})")

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Guess (AUC = 0.5000)')
plt.fill_between(fpr, tpr, alpha=0.05, color='#3498db')
plt.xlabel('False Positive Rate', fontsize=11)
plt.ylabel('True Positive Rate', fontsize=11)
plt.title('ROC Curve — All Models', fontsize=13, fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()
print('💡 A model closer to the top-left corner is better. AUC = 1.0 is perfect!')

In [ ]:
# ---- 8.4 Feature Importance (only for tree-based models) ----
best_model_obj = best['model']

if hasattr(best_model_obj, 'feature_importances_'):
    importance_df = pd.DataFrame({
        'Feature'   : X.columns,
        'Importance': best_model_obj.feature_importances_
    }).sort_values('Importance', ascending=False)

    plt.figure(figsize=(10, 6))
    colors_fi = ['#e74c3c' if i < 5 else '#3498db'
                 for i in range(len(importance_df))]
    bars = plt.barh(importance_df['Feature'][::-1],
                    importance_df['Importance'][::-1],
                    color=colors_fi[::-1])
    plt.xlabel('Importance Score')
    plt.title(f'Feature Importance — {best_name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print('🔴 Top 5 most important features:')
    print(importance_df.head(5).to_string(index=False))
else:
    print('ℹ️ Feature importance not available for Logistic Regression directly.')

## Step 9 — Make Predictions on New Transactions

Let's see how our model would work on a **brand new, unseen transaction**.

In [ ]:
# Take 5 random transactions from test set and show predictions
sample = X_test.sample(5, random_state=7)

# Use best model for prediction
X_sample = X_test_scaled[sample.index] if best_name == 'Logistic Regression' else sample

sample_pred  = best_model_obj.predict(X_sample)
sample_proba = best_model_obj.predict_proba(X_sample)[:, 1]

result_df = pd.DataFrame({
    'Actual'          : y_test.loc[sample.index].values,
    'Predicted'       : sample_pred,
    'Fraud_Probability': [f'{p*100:.1f}%' for p in sample_proba],
    'Correct?'        : ['✅' if a == p else '❌'
                         for a, p in zip(y_test.loc[sample.index].values, sample_pred)]
})

print('🔮 Sample Predictions:')
result_df

---

## 🏁 Conclusion

### What We Did

| Step | Action | Outcome |
|---|---|---|
| 1 | Loaded 100K retail transactions | Clean dataset, no missing values |
| 2 | EDA — charts & distributions | Found key patterns in fraud behaviour |
| 3 | Feature Engineering | Created 6 new features (time, risk score, amount ratio) |
| 4 | Trained 3 ML models | Logistic Regression, Random Forest, Gradient Boosting |
| 5 | Evaluated with Accuracy & AUC | Best model achieves strong performance |

### Key Findings 🔍

- **Previous fraud flag** and **velocity flag** are the strongest indicators of fraud
- Fraud transactions tend to have **higher amounts** than legitimate ones
- **International transactions** and **high-risk devices** show elevated fraud rates
- Our best model achieves **strong ROC-AUC**, meaning it reliably separates fraud from legitimate transactions

### Business Impact 💼

> If this model was deployed on 100,000 transactions, it could **automatically flag thousands of fraudulent transactions**, reducing losses significantly while keeping friction low for genuine customers.

### What's Next? 🚀

- Try **XGBoost / LightGBM** for potentially higher accuracy
- **Hyperparameter tuning** with GridSearchCV
- **SHAP values** for explainable AI — understand *why* a transaction was flagged
- Deploy the model as a **real-time API** for live transaction scoring

---
*If you found this notebook helpful, please give it an upvote ⬆️ — it encourages beginner-friendly content!*